# Session 2: Strands Agents를 활용한 Agent 구축

### Index
1. LLM vs Agent — 개념 이해
2. Strands Agents 기본 사용
3. Tool 정의 및 활용
4. 실전 예제: 여러 Tool 조합
5. 단기 메모리(Short-term Memory) 확인
6. Session 관리 — 대화 분리 및 복원
7. Agent를 AgentCore에 배포하기

---
## 1. LLM vs Agent

### LLM (Session 1에서 한 것)
- 질문 → 답변 (1회성)
- 외부 정보 접근 불가
- 실시간 데이터 모름

### Agent
- 목표를 주면 **스스로 판단**하고 **도구(Tool)를 사용**해서 해결
- 필요하면 여러 단계를 거쳐 작업 수행
- ReAct 패턴: **생각(Reasoning) → 행동(Action) → 관찰(Observation)** 반복

```
사용자: "서울 날씨 알려주고 화씨로 변환해줘"

Agent 내부:
  [생각] 서울 날씨를 먼저 조회해야겠다 → get_weather 호출
  [행동] get_weather("서울") → "맑음, 23도"
  [관찰] 섭씨 23도를 화씨로 변환해야 한다
  [행동] calculator("23 * 9/5 + 32") → "73.4"
  [답변] 서울은 맑음이고 23°C(73.4°F)입니다.
```

> 💡 Agent = LLM + Tool + 판단 루프

---
## 2. Strands Agents 기본 사용

> [Strands Agents](https://github.com/strands-agents/sdk-python)는 AWS에서 만든 오픈소스 Agent 프레임워크입니다.  
> Bedrock 모델과 자연스럽게 연동되며, 간단한 코드로 Agent를 만들 수 있습니다.

In [ ]:
# ! pip install strands-agents

In [ ]:
from strands import Agent
from strands.models import BedrockModel

In [ ]:
model = BedrockModel(
  model_id="us.amazon.nova-2-lite-v1:0",
  region_name="us-east-1"
)

### 기본 Agent 생성 및 대화

In [ ]:
agent = Agent(model=model)
response = agent("안녕하세요! 당신은 누구인가요?")

### System Prompt로 역할 부여

> Agent에 성격과 역할을 부여하면 일관된 응답을 생성합니다.

In [ ]:
agent = Agent(
  model=model,
  system_prompt="너는 AWS 전문 솔루션즈 아키텍트야. 모든 질문에 AWS 서비스를 활용한 답변을 해줘. 한국어로 10문장 이내로 답변해."
)
response = agent("사용자 인증 시스템을 만들고 싶어")

---
## 3. Tool 정의 및 활용

> Agent가 외부 세계와 상호작용하려면 **Tool(도구)**이 필요합니다.  
> Strands에서는 `@tool` 데코레이터로 간단하게 정의할 수 있습니다.

### Tool 작성 규칙
1. `@tool` 데코레이터 붙이기
2. **docstring 필수** — Agent가 이 설명을 보고 언제 사용할지 판단
3. 파라미터에 **타입 힌트** 필수 — Agent가 어떤 값을 넣을지 결정

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

In [ ]:
@tool
def get_weather(city: str) -> str:
  """도시의 현재 날씨를 조회합니다. city는 도시 이름입니다."""
  # 실제로는 외부 API를 호출하겠지만, 여기서는 하드코딩
  weather_data = {
      "서울": "맑음, 23°C",
      "부산": "흐림, 19°C",
      "제주": "비, 21°C"
  }
  return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

@tool
def calculator(expression: str) -> str:
  """수학 계산을 수행합니다. expression은 계산할 수식입니다. 예: '2 + 3', '100 * 0.15'"""
  try:
      result = eval(expression)
      return str(result)
  except Exception as e:
      return f"계산 오류: {e}"

### Tool을 Agent에 연결

In [ ]:
model = BedrockModel(
  model_id="us.amazon.nova-2-lite-v1:0",
  region_name="us-east-1"
)

agent = Agent(
  model=model,
  tools=[get_weather, calculator],
  system_prompt="너는 친절한 비서야. 도구를 활용해서 정확한 정보를 제공해. 한국어로 답변해."
)

### Tool 호출 테스트

> Agent가 질문을 분석하고, 적절한 Tool을 선택하여 호출하는 과정을 관찰합니다.

In [ ]:
# 날씨 Tool 호출
response = agent("서울 날씨 어때?")

In [ ]:
# 계산 Tool 호출
response = agent("15% 팁 포함해서 45000원짜리 식사 총 금액은?")

In [ ]:
# 여러 Tool 조합 호출
response = agent("서울 날씨 알려주고, 현재 온도를 화씨로 변환해줘")

---
## 4. 실전 예제: 여러 Tool 조합

> 좀 더 현실적인 시나리오로 Agent를 구성해봅니다.  
> **시나리오: 운동 코치 비서** — 운동 기록, 주간 요약, 목표 설정 및 달성 점검이 가능한 Agent
>
> 💡 Strands Agent는 대화 히스토리(memory)를 자체 관리합니다.  
> 별도 메모 기능 없이도 이전 대화 내용을 기억합니다.

In [ ]:
import json
from strands import Agent, tool
from strands.models import BedrockModel
from datetime import date as _date

# ── 메모리 저장소 ──
WORKOUTS = []                 # [{"date","type","minutes"}, ...]
WEEKLY_GOAL = {"minutes": 150}   # 주간 목표 (WHO 권장 기본값)

@tool
def get_today() -> str:
  """오늘 날짜와 이번 주 시작일(월요일)을 반환한다.

  사용자가 '오늘', '어제', '이번 주' 같은 상대적 표현을 쓰면
  먼저 이 툴로 기준 날짜를 확인한 뒤 다른 툴에 넘긴다.
  """
  today = _date.today()
  monday = today.fromordinal(today.toordinal() - today.weekday())  # 이번 주 월요일
  return f"오늘은 {today.isoformat()} ({'월화수목금토일'[today.weekday()]}요일), 이번 주 시작(월)은 {monday.isoformat()}"

@tool
def log_workout(date: str, workout_type: str, minutes: int) -> str:
  """운동 기록을 추가한다.

  '오늘 30분 달렸어', '어제 요가 1시간' 처럼 운동한 내역을 기록할 때 사용한다.

  Args:
      date: 운동 날짜 'YYYY-MM-DD'
      workout_type: 운동 종류 (예: "달리기", "요가", "헬스")
      minutes: 운동 시간(분), 정수
  """
  WORKOUTS.append({"date": date, "type": workout_type, "minutes": minutes})
  return f"💪 기록 완료: {date} {workout_type} {minutes}분"

@tool
def weekly_summary(start_date: str) -> str:
  """특정 주(7일)의 운동 기록과 총 운동 시간을 요약한다.

  '이번 주 얼마나 운동했어?' 처럼 한 주 운동량을 모아 볼 때 사용한다.

  Args:
      start_date: 그 주의 시작 날짜 'YYYY-MM-DD' (이 날부터 7일간 집계)
  """
  from datetime import date as d
  y, m, day = map(int, start_date.split("-"))
  base = d(y, m, day)
  items = [w for w in WORKOUTS
           if 0 <= (d(*map(int, w["date"].split("-"))) - base).days < 7]
  total = sum(w["minutes"] for w in items)
  detail = ", ".join(f"{w['type']} {w['minutes']}분" for w in items) or "기록 없음"
  return f"{start_date}부터 7일: 총 {total}분 ({detail})"

@tool
def set_goal(minutes: int) -> str:
  """주간 운동 목표 시간(분)을 설정한다.

  '주 200분 목표로 잡아줘' 처럼 목표를 정할 때 사용한다.

  Args:
      minutes: 주간 목표 운동 시간(분)
  """
  WEEKLY_GOAL["minutes"] = minutes
  return f"🎯 주간 목표를 {minutes}분으로 설정했어요."

@tool
def check_goal(start_date: str) -> str:
  """그 주의 운동량이 주간 목표를 채웠는지 확인한다.

  '목표 채웠어?', '얼마나 남았어?' 처럼 목표 대비 진행률을 볼 때 사용한다.

  Args:
      start_date: 그 주의 시작 날짜 'YYYY-MM-DD'
  """
  from datetime import date as d
  y, m, day = map(int, start_date.split("-"))
  base = d(y, m, day)
  total = sum(w["minutes"] for w in WORKOUTS
              if 0 <= (d(*map(int, w["date"].split("-"))) - base).days < 7)
  goal = WEEKLY_GOAL["minutes"]
  remain = goal - total
  if remain <= 0:
      return f"🎉 목표 달성! {total}/{goal}분 ({-remain}분 초과 달성)"
  return f"{total}/{goal}분, 목표까지 {remain}분 남았어요. 화이팅!"

SYSTEM_PROMPT = """너는 친근한 운동 코치 비서야.

처음 대화를 시작하면 할 수 있는 일을 간단히 소개해:
- 운동 기록, 주간 운동 요약, 목표 설정, 목표 달성 여부 점검

매 대화 끝에 이번 주 목표 점검을 자연스럽게 한 번 언급해줘.
- 운동 기록 → log_workout, 주간 요약 → weekly_summary, 목표 설정 → set_goal, 목표 점검 → check_goal.
- '목표 채웠어?' 같은 질문은 그 주 운동량을 모아 목표와 비교해서 답해.
- 한국어로 짧고 응원하는 톤으로 답해."""

In [ ]:
model = BedrockModel(
  model_id="us.amazon.nova-2-lite-v1:0",
  region_name="us-east-1"
)

agent = Agent(
  model=model,
  system_prompt=SYSTEM_PROMPT,
  tools=[get_today, log_workout, weekly_summary, set_goal, check_goal],
  callback_handler=None
)

In [ ]:
response = agent("안녕")
print(response)

---
## 5. 단기 메모리(Short-term Memory) 확인

> Strands Agent는 **대화 히스토리를 자동으로 관리**합니다.  
> 같은 Agent 인스턴스에 여러 번 말을 걸면, 이전 대화 내용이 모두 LLM에 전달됩니다.  
> 이를 통해 Agent는 문맥을 유지하며 연속적인 대화가 가능합니다.
>
> 아래 실습에서 3번 연속 호출하면서:
> 1. `agent.messages` — 실제 LLM에 전달되는 메시지 배열 확인
> 2. 토큰 사용량 — 대화가 쌓일수록 input 토큰이 증가하는 것을 확인

In [ ]:
from strands import Agent
from strands.models import BedrockModel

model = BedrockModel(
  model_id="us.amazon.nova-2-lite-v1:0",
  region_name="us-east-1"
)

agent = Agent(
  model=model,
  system_prompt="너는 친절한 비서야. 한국어로 간결하게 답변해."
)

print("=" * 50)
print("Agent 생성 완료. 대화 시작 전 메시지 수:", len(agent.messages))
print("=" * 50)

In [ ]:
# 1번째 대화
response = agent("나는 오늘 점심으로 피자를 먹었어.")

print("\n" + "=" * 50)
print(f"[1번째 호출 후] 메시지 수: {len(agent.messages)}")
print("=" * 50)

In [ ]:
# 2번째 대화
response = agent("오늘 저녁은 햄버거를 먹었어.")

print("\n" + "=" * 50)
print(f"[2번째 호출 후] 메시지 수: {len(agent.messages)}")
print("=" * 50)

In [ ]:
# 3번째 대화 — 이전 대화 내용을 기억하는지 확인
response = agent("내가 오늘 뭘 먹었다고 했어?")

print("\n" + "=" * 50)
print(f"[3번째 호출 후] 메시지 수: {len(agent.messages)}")
print("=" * 50)

In [ ]:
# 이 셀을 여러 번 반복 실행해보세요 — inputTokens가 점점 증가합니다!
response = agent("내일 아침으로 뭘먹을까?")
print(f"✅ inputTokens: {response.metrics.accumulated_usage['inputTokens']}")

---
## 6. Session 관리 — 대화 분리 및 복원

> 단기 메모리는 같은 Agent 인스턴스 내에서만 유지됩니다.  
> 프로그램을 재시작하면 대화 히스토리가 사라집니다.
>
> **SessionManager**를 사용하면:
> - 대화를 **영구 저장**할 수 있고
> - **session_id**로 서로 다른 대화를 **분리**할 수 있습니다.
>
> 아래 실습에서 두 개의 세션을 만들어 각각 다른 대화를 진행해봅니다.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.session.file_session_manager import FileSessionManager

model = BedrockModel(
  model_id="us.anthropic.claude-sonnet-4-6",
  region_name="us-east-1"
)

# 세션 A: 민수의 대화
session_a = FileSessionManager(session_id="user-a", storage_dir="./sessions")
agent_a = Agent(model=model, session_manager=session_a, system_prompt="한국어로 간결하게 답변해. 답변은 2문장 이내로.", callback_handler=None)

# 세션 B: 영희의 대화
session_b = FileSessionManager(session_id="user-b", storage_dir="./sessions")
agent_b = Agent(model=model, session_manager=session_b, system_prompt="한국어로 간결하게 답변해. 답변은 2문장 이내로.", callback_handler=None)

print("✅ 두 개의 세션이 생성되었습니다: user-a, user-b")

In [ ]:
# 세션 A에서 대화
print(agent_a("다음 달에 제주도 3박 4일 여행 계획 중이야."))
print(f"\n[세션 A] 메시지 수: {len(agent_a.messages)}")

In [ ]:
# 세션 B에서 대화
print(agent_b("이번 주말에 부산 당일치기 계획 중이야."))
print(f"\n[세션 B] 메시지 수: {len(agent_b.messages)}")

In [ ]:
# 각 세션이 독립적인지 확인
print("--- 세션 A에게 질문 ---")
print(agent_a("내가 어디 여행 간다고 했지?"))

print("\n--- 세션 B에게 질문 ---")
print(agent_b("내가 어디 여행 간다고 했지?"))

In [ ]:
# 세션 복원 테스트 — 새 Agent를 같은 session_id로 생성하면 이전 대화가 복원됩니다
# Session : user-a Session (제주도)
restored_session = FileSessionManager(session_id="user-a", storage_dir="./sessions")
restored_agent = Agent(model=model, session_manager=restored_session, system_prompt="한국어로 간결하게 답변해.", callback_handler=None)

print(f"복원된 메시지 수: {len(restored_agent.messages)}")
print("\n--- 복원된 세션에게 질문 ---")
print(restored_agent("내가 어디 여행 간다고 했지?"))

---
## 7. Agent를 AgentCore에 배포하기

> 이제 이 Agent를 **Amazon Bedrock AgentCore**에 배포하여 API로 호출하는 실습을 진행합니다.
>
> ### 배포 흐름
> ```
> 1. agentcore.py 작성 (Agent 코드 + 엔트리포인트)
> 2. 로컬에서 테스트 (python agentcore.py)
> 3. Docker 이미지 빌드 → ECR 업로드
> 4. AgentCore 콘솔에서 배포
> 5. SDK로 호출
> ```
>
> ⚠️ 이미지 빌드/업로드는 시간이 걸리므로, **미리 준비된 공용 ECR 이미지**를 사용합니다.

### AgentCore 콘솔에서 배포

1. [AgentCore 콘솔](https://console.aws.amazon.com/bedrock-agentcore/) 접속
2. **Runtime** → **Create** 클릭
3. 이미지 URI 입력:
   ```
   055332451319.dkr.ecr.us-east-1.amazonaws.com/aiworkshop:latest
   ```
4. 배포 완료 후 **Runtime ARN** 복사

> 📋 제공하는 이미지 URI와 Runtime ARN을 사용하세요.

### 배포된 Agent 호출하기

In [ ]:
import boto3
import json
import uuid

AWS_REGION='us-east-1'
# AGENTCORE_RUNTIME_ARN = 'arn:aws:bedrock-agentcore:<aws-region>:<aws-account-id>:runtime/<agentcore-runtime-id>'

client = boto3.client("bedrock-agentcore", region_name=AWS_REGION)

# 세션 ID 생성 (33자 이상 필수)
session_id = "session-" + uuid.uuid4().hex[:33]


response = client.invoke_agent_runtime(
    agentRuntimeArn=AGENTCORE_RUNTIME_ARN,
    runtimeSessionId=session_id, # Must be 33+ char. Every new SessionId will create a new MicroVM
    payload=json.dumps({"prompt": "오늘 러닝 30분 했어."}).encode()
)

# 응답 출력
response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)

## Agent Session과 AgentCore Runtime Session의 차이

### Strands Agent Session (SDK 레벨)

>FileSessionManager로 대화 히스토리(메시지 배열)를 파일에 저장  
>같은 session_id로 Agent를 재생성하면 이전 대화 내용 복원  
>"LLM이 뭘 기억하는가"에 관한 것  

### AgentCore Session (인프라 레벨)

>runtimeSessionId로 어느 microVM으로 요청을 라우팅할지 결정  
>같은 ID → 같은 microVM → 메모리에 살아있는 같은 Agent 인스턴스  
>다른 ID → 다른 microVM → 새 Agent 인스턴스  
>"어떤 프로세스가 처리하는가"에 관한 것  

In [ ]:
# 같은 세션으로 추가 질문 (대화 유지)
response = client.invoke_agent_runtime(
    agentRuntimeArn=AGENTCORE_RUNTIME_ARN,
    runtimeSessionId=session_id,  # 같은 세션 ID → 같은 microVM
    payload=json.dumps({"prompt": "이번주 운동 기록 알려주고, 주간 목표 몇분 남았는지 알려줘"}).encode()
)

response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)

In [ ]:
# 다른 세션으로 추가 질문 (대화 단절)

session_id = "session-" + uuid.uuid4().hex[:33]

response = client.invoke_agent_runtime(
    agentRuntimeArn=AGENTCORE_RUNTIME_ARN,
    runtimeSessionId=session_id,  # 다른 세션 ID → 다른 microVM
    payload=json.dumps({"prompt": "이번주 운동 기록 알려주고, 주간 목표 몇분 남았는지 알려줘"}).encode()
)

response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)